**🔹 1. Install Required Libraries**

In [1]:
!pip install transformers datasets accelerate torch


**🔹 2. Import Libraries**

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)


**🔹 3. Load SST-2 Dataset**

In [3]:
dataset = load_dataset("glue", "sst2")

print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


**🔹 4. Load Tokenizer**

In [5]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

**🔹 5. Tokenize Dataset**

In [6]:
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(["sentence", "idx"])
tokenized_dataset.set_format("torch")


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

**🔹 6. Load DistilBERT Model**

In [7]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**🔹 7. Training Arguments (FAST & SAFE)**

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

**🔹 8. Trainer Setup**

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2463689576.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


**🔹 9. Train the Model**

In [11]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.163100,0.317972
2,0.111700,0.360532


TrainOutput(global_step=8420, training_loss=0.17636826683959597, metrics={'train_runtime': 1445.2374, 'train_samples_per_second': 93.201, 'train_steps_per_second': 5.826, 'total_flos': 4460773416041472.0, 'train_loss': 0.17636826683959597, 'epoch': 2.0})

**⚠️ We use 2 epochs → fast + sufficient for explainability.**

**🔹 10. Evaluate Model**

In [12]:
eval_results = trainer.evaluate()
print(eval_results)


{'eval_loss': 0.3179716169834137, 'eval_runtime': 3.0797, 'eval_samples_per_second': 283.141, 'eval_steps_per_second': 17.859, 'epoch': 2.0}


**🔹 11. Save Model & Tokenizer**

In [13]:
model.save_pretrained("./distilbert_sst2")
tokenizer.save_pretrained("./distilbert_sst2")


('./distilbert_sst2/tokenizer_config.json',
 './distilbert_sst2/special_tokens_map.json',
 './distilbert_sst2/vocab.txt',
 './distilbert_sst2/added_tokens.json',
 './distilbert_sst2/tokenizer.json')

**🔹 12.Check & Download into local system**

In [15]:
!ls distilbert_sst2


config.json	   special_tokens_map.json  tokenizer.json
model.safetensors  tokenizer_config.json    vocab.txt


In [17]:
from google.colab import files

files.download("distilbert_sst2/config.json")
files.download("distilbert_sst2/model.safetensors")
files.download("distilbert_sst2/special_tokens_map.json")
files.download("distilbert_sst2/tokenizer.json")
files.download("distilbert_sst2/tokenizer_config.json")
files.download("distilbert_sst2/vocab.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

****Testing Metric Accuracy****

In [20]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00


In [23]:
import numpy as np
from evaluate import load

In [24]:
metric = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-217095029.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [26]:
eval_results = trainer.evaluate()
print(eval_results)


{'eval_loss': 0.3179716169834137, 'eval_model_preparation_time': 0.0015, 'eval_accuracy': 0.8990825688073395, 'eval_runtime': 3.4777, 'eval_samples_per_second': 250.741, 'eval_steps_per_second': 15.815}
